# Problem Set 2
**ECON 422: Macroeconomics & Machine Learning**

**Instructions.** Answer all four problems. Each problem is worth **2.5 points: 0.5 basic credit for a meaningful attempt and 1 point for each of the two subquestions** (10 points total). Give short explanations and complete the marked code. Submit your notebook with outputs.

## Data and Python setup

Problems 2–3 use monthly FRED retail sales from 2000–2025. Estimate using earlier observations and evaluate forecasts in 2025. Use unadjusted sales for Problem 2 and seasonally adjusted sales for Problem 3. These are nominal sales in millions of dollars. Problem 4 uses supplied simulated data.

Use observations dated before each forecast month. These are retrospective forecasts using revised data, not a real-time release-calendar exercise.

In [ ]:
# Run once if needed, then restart the kernel.
%pip install -q numpy pandas matplotlib scikit-learn statsmodels requests

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from getpass import getpass
from pathlib import Path
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from statsmodels.tsa.stattools import adfuller
from IPython.display import display

plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True,
                     'axes.prop_cycle': plt.cycler(color=['#80b98a','#345c48','#d3a453','#748cb5'])})
SEED = 422

def rmse(actual, predicted):
    return float(np.sqrt(np.mean((np.asarray(actual)-np.asarray(predicted))**2)))


### Shared FRED setup (provided)
Obtain a free [FRED API key](https://fred.stlouisfed.org/docs/api/api_key.html). Type or paste it into the hidden prompt when the next cell runs. No key is stored in the notebook. A CSV cache allows later reruns without another request; delete the cache to refresh it.

| Column | FRED ID | Description |
|---|---|---|
| NSA | RSXFSN | Retail trade sales, not seasonally adjusted, millions of dollars |
| SA | RSXFS | Retail trade sales, seasonally adjusted, millions of dollars |

Use the same downloaded file for Problems 2–3. Problem 4 uses supplied simulated data. Missing observations cause an explicit error; do not silently interpolate them.

In [ ]:
CACHE = Path('retail_FRED_2000_2025.csv')
if CACHE.exists():
    data = pd.read_csv(CACHE, index_col=0, parse_dates=True)
else:
    key = getpass('Paste your FRED API key (hidden): ').strip()
    def fetch_series(series_id):
        try:
            response = requests.get(
                'https://api.stlouisfed.org/fred/series/observations',
                params={'series_id': series_id, 'api_key': key, 'file_type': 'json',
                        'observation_start': '2000-01-01', 'observation_end': '2025-12-31'},
                timeout=45)
        except requests.RequestException:
            raise RuntimeError('FRED connection failed. Check internet access and retry.') from None
        if response.status_code != 200:
            raise RuntimeError('FRED request failed. Check your API key and try again.')
        rows = response.json()['observations']
        return pd.Series([float(r['value']) if r['value'] != '.' else np.nan for r in rows],
                         index=pd.to_datetime([r['date'] for r in rows]))
    try:
        data = pd.DataFrame({'NSA': fetch_series('RSXFSN'), 'SA': fetch_series('RSXFS')})
    finally:
        del key
    data.to_csv(CACHE, index_label='date')
expected = pd.date_range('2000-01-01','2025-12-01',freq='MS')
data = data.reindex(expected)[['NSA','SA']].astype(float)
assert data.notna().all().all(), 'Missing months: refresh the cache or inspect FRED availability.'
assert (data > 0).all().all(), 'Logarithms require positive sales.'
data.index.name = 'date'
display(data.head(), data.tail())
print('Months:', len(data), '| Forecast evaluation months:', sum(data.index.year == 2025))


## Problem 1. Time series and machine learning

**(a)** What is a time series? Give one economic example and explain why the order of observations matters.

**Answer (a):**

**(b)** In your view, what role should classical time-series models play alongside ML in forecasting? Should they be replaced by ML, used together with ML, or used for different purposes? Briefly explain your reasoning.

**Answer (b):**

## Problem 2. Trend and seasonality: forecasting 2025

**(a)** Using unadjusted sales from 2000–2024, plot sales over time and average sales by calendar month. Briefly explain trend and seasonality.

In [ ]:
# TODO Q2(a): plot earlier sales and calendar-month averages.
earlier = data.loc[:'2024-12-01', 'NSA']


**Answer (a):**

**(b)** Complete the supplied code to forecast each month of 2025 using
$$
\log(S_t)=\beta_0+\beta_1t+\sum_{j=2}^{12}\delta_jD_{jt}+u_t.
$$
Use only observations before the forecast month. Convert log forecasts with `np.exp`, without a bias adjustment. Compare with the seasonal-naive forecast $\widehat S_t=S_{t-12}$. Report both RMSEs in millions of dollars, plot actual and predicted sales, and state which method performs better in 2025.

In [ ]:
# Provided calendar design: its entries are known before outcomes occur.
calendar = pd.DataFrame({'time': np.arange(len(data), dtype=float)}, index=data.index)
for month in range(2,13):
    calendar[f'month_{month}'] = (data.index.month == month).astype(float)
test_dates = data.index[data.index.year == 2025]


In [ ]:
q2 = pd.DataFrame(index=test_dates)
q2['Actual'] = data.loc[test_dates,'NSA']
for date in test_dates:
    past = data.index < date
    # TODO fit LinearRegression on calendar.loc[past] and log sales.
    # TODO store the exponentiated forecast and sales from 12 months earlier.
    pass
# TODO report RMSE and plot the forecasts.


**Answer (b):**

## Problem 3. Autoregression and a unit root

**(a)** For $z_t=a+\rho z_{t-1}+e_t$, explain how a shock behaves when $|\rho|<1$ and when $\rho=1$. Why might we use growth instead of the level?

**Answer (a):**

**(b)** Using seasonally adjusted sales, let $r_t=100[\log(S_t^{SA})-\log(S_{t-1}^{SA})]$. Complete the code to forecast each month of 2025 with $r_t=a+\phi r_{t-1}+v_t$, using only preceding observations. Compare with a zero-growth forecast. Report both RMSEs in percentage points, plot the forecasts, and briefly interpret the fitted lag coefficient.

In [ ]:
z = np.log(data['SA'])
r = (100*z.diff()).rename('growth')

In [ ]:
ar_data = pd.concat([r, r.shift(1).rename('lag1')],axis=1).dropna()


In [ ]:
q3 = pd.DataFrame(index=test_dates)
q3['Actual'] = r.loc[test_dates]
q3['Zero growth'] = 0.0
for date in test_dates:
    train = ar_data.loc[ar_data.index < date]
    # TODO fit growth on lag1 and forecast this month's growth.
    pass
# TODO report RMSEs, plot, and report the fitted lag coefficient.


**Answer (b):**

## Problem 4. Reversibility and time-series DML

Use the supplied simulation for this problem:
$$
X_t=0.5X_{t-1}+u_t,\qquad D_t=X_t+v_t,\qquad Y_t=2D_t+X_t+e_t.
$$
All shocks are independent standard normal draws across dates and types. Start with $X_0\sim N(0,4/3)$, independently of the shocks. The target coefficient on D is 2.

**Given fact:** the stationary Gaussian AR(1) X is reversible. Adding independent noise and applying the same transformation at each date makes $(Y_t,D_t,X_t)$ reversible too.

**Provided procedure:** split 600 observations into A (1–300) and B (301–600). Fit on B in reverse row order and evaluate on A; then fit on A and evaluate on B. Keep each row together. Average the two residual slopes. This is full-sample coefficient estimation, not forecasting. Reversibility alone does not guarantee valid inference; this example also uses independent errors and a stable AR process.

**(a)** Explain reversibility in your own words. Why does reversing a sequence preserve the distances between dates, while randomly shuffling it generally does not? Does reversibility mean the realized sequence must read identically backward?

**Answer (a):**

**(b)** Run the supplied simulation and complete the four marked lines in `reverse_dml`. Fit LASSO predictions of Y and D from X on the auxiliary block, then calculate residuals on the other block:
$$
\widetilde Y_t=Y_t-\widehat\ell(X_t),\qquad
\widetilde D_t=D_t-\widehat m(X_t),\qquad
\widehat\theta_k=\frac{\sum_{t\in k}\widetilde D_t\widetilde Y_t}{\sum_{t\in k}\widetilde D_t^2}.
$$
Report the estimates for blocks A and B and their average. Compare the average with the true coefficient 2. No standard errors are required.

In [ ]:
# Provided simulation; Problem 4 can run independently of FRED.
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(422)
n = 600
x = np.empty(n)
x_previous = rng.normal(scale=np.sqrt(4/3))
for t in range(n):
    x[t] = 0.5*x_previous + rng.normal()
    x_previous = x[t]
v = rng.normal(size=n)
e = rng.normal(size=n)
d = x + v
y = 2*d + x + e
toy = pd.DataFrame({'X': x, 'D': d, 'Y': y})


In [ ]:
def reverse_dml(frame):
    n = len(frame)
    assert n % 2 == 0
    half = n // 2
    # (main indices, auxiliary indices, direction)
    schedule = [
        (np.arange(half), np.arange(n-1, half-1, -1), 'reverse'),
        (np.arange(half, n), np.arange(half), 'forward')]
    rows = []
    for label, (main_idx, aux_idx, direction) in zip(['A', 'B'], schedule):
        assert not np.intersect1d(main_idx, aux_idx).size
        aux = frame.iloc[aux_idx]
        main = frame.iloc[main_idx]
        alpha = 1.0 / len(aux)
        model_y = make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=10000))
        model_d = make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=10000))
        model_y.fit(aux[['X']], aux['Y'])
        model_d.fit(aux[['X']], aux['D'])
        y_res = ...  # TODO 1: outcome residual
        d_res = ...  # TODO 2: regressor residual
        theta = ...  # TODO 3: block residual slope
        rows.append({'Main block': label, 'Auxiliary direction': direction,
                     'Estimate': theta, 'Mean squared D residual': np.mean(d_res**2)})
    table = pd.DataFrame(rows)
    average = ...  # TODO 4: average the two block slopes
    return table, average


In [ ]:
# Run after completing the four TODO lines.
block_results, theta_dml = reverse_dml(toy)
print(block_results.to_string(index=False))
print(f'Average reverse-cross-fitted estimate: {theta_dml:.4f}')
print(f'Known coefficient: {2.0:.4f}')


**Answer (b):**